In [52]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/data-science-competition-dscb-ss-26/sample_submission.csv
/kaggle/input/competitions/data-science-competition-dscb-ss-26/train.csv
/kaggle/input/competitions/data-science-competition-dscb-ss-26/test.csv


In [53]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [54]:
# Daten laden
# Daten laden
train_path = "/kaggle/input/competitions/data-science-competition-dscb-ss-26/train.csv"
test_path = "/kaggle/input/competitions/data-science-competition-dscb-ss-26/test.csv"
sample_submission_path = "/kaggle/input/competitions/data-science-competition-dscb-ss-26/sample_submission.csv"
output_path = "/kaggle/working/submission.csv"

train = pd.read_csv(train_path, parse_dates=["date"])
test = pd.read_csv(test_path, parse_dates=["date"])
sample_submission = pd.read_csv(sample_submission_path)

# Kalender Features
train["month"] = train["date"].dt.month
train["weekday"] = train["date"].dt.weekday
test["month"] = test["date"].dt.month
test["weekday"] = test["date"].dt.weekday

print("Train:", train.shape, " Test:", test.shape)

Train: (14319, 17)  Test: (2845, 16)


In [55]:
# Feature Liste festlegen
categorical_features = ["station", "wd"]
numeric_features = [
    "pm10", "so2", "no2", "co", "o3",
    "temp", "pres", "dewp", "rain", "wspm",
    "month", "weekday",
]
feature_columns = categorical_features + numeric_features

In [56]:
# Validation Split
validation_start = pd.Timestamp("2016-01-01")
validation_end   = pd.Timestamp("2016-06-30")

fit_mask   = train["date"] < validation_start
valid_mask = (train["date"] >= validation_start) & (train["date"] <= validation_end)

X_fit   = train.loc[fit_mask,   feature_columns]
y_fit   = train.loc[fit_mask,   "pm25"]
X_valid = train.loc[valid_mask, feature_columns]
y_valid = train.loc[valid_mask, "pm25"]

print("X_fit:", X_fit.shape, " X_valid:", X_valid.shape)

X_fit: (12166, 14)  X_valid: (2153, 14)


In [57]:
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
    ]
)

In [58]:
# Linear Baseline
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()),
])
linear_model.fit(X_fit, y_fit)
linear_valid_pred = linear_model.predict(X_valid)
linear_valid_mse  = mean_squared_error(y_valid, linear_valid_pred)

In [59]:
regularized_linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge()),
])

In [60]:
# Regularisierte Suche
regularized_linear_search = RandomizedSearchCV(
    estimator=regularized_linear_pipeline,
    param_distributions=[
        {"regressor": [Ridge()],      "regressor__alpha": [0.01, 0.1, 1.0, 10.0]},
        {"regressor": [Lasso(max_iter=10000)], "regressor__alpha": [0.01, 0.1, 1.0, 10.0]},
        {"regressor": [ElasticNet(max_iter=10000)],
         "regressor__alpha": [0.01, 0.1, 1.0, 10.0],
         "regressor__l1_ratio": [0.2, 0.5, 0.8]},
    ],
    n_iter=25,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="neg_mean_squared_error",
    random_state=42, n_jobs=1,
)
regularized_linear_search.fit(X_fit, y_fit)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 20 is smaller than n_iter=25. Running 20 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=None),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('categorical',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='most_frequent')),
                                                                                               ('encoder',
                                                                                                OneHotEncoder(handle_unknown='ignore'))]),
                                                                               ['station',
                                                                                'wd']),
                                                                              ('numeric',
                                                                               Pipeline(steps=[('im...
                                             ('regressor', Ridge())]),
                   n_iter=25, n_jobs=1,
                   param_distributions=[{'regressor': [Ridge()],
                                         'regressor__alpha': [0.01, 0.1, 1.0,
                                                              10.0]},
                                        {'regressor': [Lasso(max_iter=10000)],
                                         'regressor__alpha': [0.01, 0.1, 1.0,
                                                              10.0]},
                                        {'regressor': [ElasticNet(max_iter=10000)],
                                         'regressor__alpha': [0.01, 0.1, 1.0,
                                                              10.0],
                                         'regressor__l1_ratio': [0.2, 0.5,
                                                                 0.8]}],
                   random_state=42, scoring='neg_mean_squared_error')

In [61]:
regularized_linear_model = regularized_linear_search.best_estimator_
regularized_linear_valid_pred = regularized_linear_model.predict(X_valid)
regularized_linear_valid_mse  = mean_squared_error(y_valid, regularized_linear_valid_pred)
print(f"Regularisiert Validation-MSE: {regularized_linear_valid_mse:.3f}")

X_train_full = train[feature_columns]
y_train_full = train["pm25"]
X_test       = test[feature_columns]

Regularisiert Validation-MSE: 444.989


In [62]:
# Modell suchen
if regularized_linear_valid_mse < linear_valid_mse:
    best_model = regularized_linear_model
else:
    best_model = linear_model

best_model.fit(X_train_full, y_train_full)
test_pred = best_model.predict(X_test)

In [63]:
# Submission erzeugen + speichern
predictions_by_id = pd.Series(test_pred, index=test["id"])

submission = sample_submission[["id"]].copy()
raw_pred = predictions_by_id.reindex(submission["id"]).to_numpy()

# Clipping: PM2.5 kann nicht negativ sein
min_pm25 = float(train["pm25"].min())
submission["pm25"] = np.round(np.clip(raw_pred, a_min=min_pm25, a_max=None), 3)

submission.to_csv(output_path, index=False)
print("Submission geschrieben:", output_path)
print("Shape:", submission.shape)
submission.head()

Submission geschrieben: /kaggle/working/submission.csv
Shape: (2845, 2)


,id,pm25
0,Aotizhongxin_2016-07-01,3.000
1,Changping_2016-07-01,3.000
2,Dingling_2016-07-01,3.000
3,Dongsi_2016-07-01,6.351
4,Guanyuan_2016-07-01,6.644
